# Secure Payment Agent (Stripe/Privy) — end-to-end test notebook

Drives every script in this sample in order, against **Base Sepolia testnet only**.
Testnet USDC has no value. Nothing here runs on mainnet.

**Before you start**
1. `cp .env.privy.sample .env` and fill in your AWS + Privy values.
2. Have AWS credentials configured (`aws sts get-caller-identity` works).
3. Have a **dedicated** Privy app (App ID/Secret + P-256 authorization key).

Each step is a cell. Steps 2–4, 6–9 call AWS/Privy and cost nothing on testnet;
Step 5 (guardrails) runs fully offline.

## Step 0 — Make the notebook's folder the working directory
All scripts use paths relative to the repo root (`setup/`, `agent/`).

In [ ]:
import os, pathlib
# When run from the privy/ folder this is a no-op; adjust if your kernel starts elsewhere.
ROOT = pathlib.Path.cwd()
if (ROOT / 'setup' / 'provision_stack.py').exists():
    print('OK working dir:', ROOT)
elif (ROOT / 'privy' / 'setup' / 'provision_stack.py').exists():
    os.chdir(ROOT / 'privy'); print('Changed into privy/:', pathlib.Path.cwd())
else:
    raise SystemExit('Run this notebook from the privy/ sample folder.')

## Step 1 — Install dependencies

In [ ]:
%pip install -q -r requirements.txt

## Step 2 — Check your .env is filled in
Confirms the file exists and flags any values still left as `<placeholders>`.
This prints only key names, never secret values.

In [ ]:
import os
from dotenv import load_dotenv
if not os.path.exists('.env'):
    raise SystemExit('No .env found. Run: cp .env.privy.sample .env  and fill it in.')
load_dotenv('.env', override=True)
required = ['AWS_REGION','PRIVY_APP_ID','PRIVY_APP_SECRET',
            'PRIVY_AUTHORIZATION_ID','PRIVY_AUTHORIZATION_PRIVATE_KEY',
            'LINKED_EMAIL','USER_ID']
missing = [k for k in required if not os.environ.get(k,'').strip() or os.environ.get(k,'').startswith('<')]
key = os.environ.get('PRIVY_AUTHORIZATION_PRIVATE_KEY','')
if key.startswith('wallet-auth:'):
    print('WARN: strip the "wallet-auth:" prefix from PRIVY_AUTHORIZATION_PRIVATE_KEY (store raw base64).')
print('Missing/placeholder:', missing or 'none — looks good')

## Step 3 — Confirm AWS credentials

In [ ]:
import boto3
print('Authenticated as:', boto3.Session().client('sts').get_caller_identity()['Arn'])

## Step 4 — Provision the shared payment stack
Creates the 4 least-privilege IAM roles, the **StripePrivy** credential provider,
the Payment Manager, and the Payment Connector. Idempotent — safe to re-run.
Writes all ARNs/IDs into `.env`.

In [ ]:
!python setup/provision_stack.py

## Step 5 — Create the per-user wallet + budgeted session
Creates a Privy embedded wallet (payment instrument) and a time-bounded, budgeted
Payment Session. Prints the wallet address and the delegation instructions.

In [ ]:
!python setup/provision_payments.py

## Step 6 — Create the Privy policy, then fund and delegate (manual, one-time)
1. Run the next cell: it creates the Privy policy and prints its ID (the wallet is user-owned, so
   the policy is attached at delegation, not by this app).
2. Fund `WALLET_ADDRESS` with testnet USDC at https://faucet.circle.com/ (Base Sepolia).
3. Set up the patched delegation frontend in a terminal:
   `bash frontend-patch/setup_frontend.sh && cd ../privy-agentcore-frontend && npx -y pnpm@9 dev`
4. Add `http://localhost:3000` to the Privy app's allowed origins, open it, log in as `LINKED_EMAIL`,
   and click **Connect agent** — this sets the policy on the agent signer.

Then run the polling cell to wait until the instrument is `ACTIVE`.

In [ ]:
!python setup/privy_policy_setup.py

In [ ]:
import time, os, boto3
from dotenv import load_dotenv; load_dotenv('.env', override=True)
from bedrock_agentcore.payments import PaymentManager
mgr = PaymentManager(payment_manager_arn=os.environ['PAYMENT_MANAGER_ARN'], region_name=os.environ['AWS_REGION'])
for _ in range(30):
    inst = mgr.get_payment_instrument(user_id=os.environ['USER_ID'], payment_instrument_id=os.environ['INSTRUMENT_ID'])
    status = inst.get('status') or inst.get('paymentInstrument',{}).get('status')
    print('instrument status:', status)
    if status == 'ACTIVE': break
    time.sleep(10)

## Step 7 — Guardrails demo (offline, no AWS/Privy calls)
Shows the deterministic fail-closed decisions: happy path approved, high amount rejected,
non-allowlisted recipient rejected. This mirrors the policy layers in-process.

In [ ]:
!python agent/guardrails_demo.py

## Step 7b — Dogwood temporal policies (offline, no AWS calls)
Validates `policies/dogwood/payment_guardrails.dw` and replays 10 scenarios with the `dogwood` CLI:
recipient provenance (block an injected or invented address), velocity limit, rolling spend cap,
and the session-reset caveat. Build the CLI first — see `policies/dogwood/README.md` — and set
`DOGWOOD` to the binary path (or put `dogwood` on PATH). Skips cleanly if the CLI is not installed.

In [ ]:
import os, shutil
# os.environ['DOGWOOD'] = '/path/to/dogwood/target/release/dogwood'
if os.environ.get('DOGWOOD') or shutil.which('dogwood'):
    !python policies/dogwood/test_dogwood_policies.py
else:
    print('dogwood CLI not found — skipping. See policies/dogwood/README.md to build it.')

## Step 8 — Live: the AgentCore Payment Session budget rejects an over-budget payment
Server-side, deterministic — the rejection is a raised `InsufficientBudget`, no LLM in the loop.

In [ ]:
!python agent/session_budget_demo.py

## Step 9 — Probe the Privy policy directly
Signs EIP-3009 messages through Privy as the AgentCore signer and checks each verdict: allowed
merchant → signed; wrong recipient, over-cap, wrong chain → denied. Sent with and without
`EIP712Domain`, because Privy matches typed-data conditions only on an exact `types` match.
Every message has `validBefore = 1` (already expired), so no signature can move funds.
If every case is SIGNED, the policy is not on the agent signer — redo the delegation in Step 6.

In [ ]:
!python setup/privy_policy_probe.py

## Step 10 — Live: the Privy policy decides whether ProcessPayment succeeds
Flips the policy's condition set (which the app owns) and drives a real x402 payment through
AgentCore each time: merchant allowed → must succeed; decoy only → must be refused; merchant
again → must succeed. Only the recipient list changes, so the refusal is the policy. AgentCore
currently reports a Privy refusal as `InternalServerException`; the third step confirms the cause.
No payment header is sent, so no USDC moves. The condition set is restored afterwards.

In [ ]:
!python setup/privy_policy_payment_demo.py

## Step 11 — Run the autonomous agent (happy path)
The agent fetches the paid endpoint, hits 402, pays in USDC via Privy, and returns the content.
Needs: funded + delegated wallet (Step 6), budget above the price, and the merchant on the allowlist.

In [ ]:
!python agent/secure_payment_agent.py

## Step 12 — Cleanup
Detach + delete the Privy policy. Optionally tear down the whole stack (uncomment).

In [ ]:
!python setup/privy_policy_remove.py
# Full teardown (optional) — deletes instrument, connector, manager:
# import os, boto3
# from dotenv import load_dotenv; load_dotenv('.env', override=True)
# cp = boto3.client('bedrock-agentcore-control', region_name=os.environ['AWS_REGION'])
# cp.delete_payment_connector(paymentManagerId=os.environ['PAYMENT_MANAGER_ID'], paymentConnectorId=os.environ['PAYMENT_CONNECTOR_ID'])
# cp.delete_payment_manager(paymentManagerId=os.environ['PAYMENT_MANAGER_ID'])